<a href="https://colab.research.google.com/github/ASaragga/GRF/blob/main/SimMonteCarlo02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import Pkg; Pkg.add("Distributions")
using Distributions, Statistics, Random

In [ ]:
 # Volatilidade Estocástica (Modelo Heston)
 function heston_simulation(S0, v0, K, r, kappa, theta, sigma_v, rho, T, dt, N_sim, days)
    N_steps = Int(days)
    S = fill(S0, N_sim)
    v = fill(v0, N_sim)
    sqrt_dt = sqrt(dt)

    for _ in 1:N_steps
        Z1 = randn(N_sim)  # Variávesi aleatórias Normais standard
        Z2 = randn(N_sim)
        W1 = Z1
        W2 = rho .* Z1 .+ sqrt(1 - rho^2) .* Z2  # Movimento Browniano correlacionado

        # Processo da variância (processo CIR)
        v = max.(0, v .+ kappa * (theta .- v) * dt .+ sigma_v * sqrt_dt * sqrt.(max.(v, 0)) .* W2)

        # Processo risco-neutral de preços
        S = S .* exp.((r .- 0.5 * max.(v, 0)) * dt .+ sqrt.(max.(v, 0)) * sqrt_dt .* W1)
    end
    return S, v
 end

In [ ]:
# Parâmetros Modelo de Heston
 S0 = 9.0    # Preço Spot
 K = 9.2     # Preço Exercício
 r = 0.03    # Taxa juro sem risco
 T = 3/12    # Tempo até ao Vencimento (em anos)
 kappa = 2.0     # Velocidade de reversão para a média da volatilidade
 theta = 0.04    # Variãncia de longo-prazo
 sigma_v = 0.3   # Volatilidade da volatilidade
 rho = -0.7      # Correlação entre os choques do preço do ativo e da volatilidade
 v0 = 0.04       # Variãncia inicial
 N_sim = 100000  # Número de simulações de Monte-Carlo
 dt = 1/252      # Passo diário (dias de negociação)

In [ ]:
 # Black-Scholes
 function black_scholes_call(S, K, r, sigma, T)
    d1 = (log(S / K) + (r + 0.5 * sigma^2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * cdf(Normal(0,1), d1) - K * exp(-r * T) * cdf(Normal(0,1), d2)
 end

In [ ]:
 # Simular 1-dia e 10-dias forward: preços e volatilidades
 S1, v1 = heston_simulation(S0, v0, K, r, kappa, theta, sigma_v, rho, T, dt, N_sim, 1)
 S10, v10 = heston_simulation(S0, v0, K, r, kappa, theta, sigma_v, rho, T, dt, N_sim, 10)

In [ ]:
# Calcular preços da opção, (i) inicial (ii) a 1-dia, e (iii) a 10-dias, usando Black-Scholes (pequena inconsistência aqui)
 C0 = black_scholes_call(S0, K, r, sqrt(v0), T)
 C1 = black_scholes_call.(S1, K, r, sqrt.(max.(v1, 0)), T - dt)
 C10 = black_scholes_call.(S10, K, r, sqrt.(max.(v10, 0)), T - 10 * dt)

In [ ]:
# Calcular a distribuição de Ganhos & Perdas
 PL1 = C1 .- C0
 PL10 = C10 .- C0

In [ ]:
# Calcular VaR e ETL a nível de significância de 5%
 VaR_1d = -quantile(PL1, 0.05)
 ETL_1d = -mean(filter(x -> x < -VaR_1d, PL1))

 VaR_10d = -quantile(PL10, 0.05)
 ETL_10d = -mean(filter(x -> x < -VaR_10d, PL10))

 println("Preço opção hoje = ", C0)
 println("1-Dia VaR(5%) = ", VaR_1d)
 println("1-Dia ETL(5%) = ", ETL_1d)
 println("10-Dias VaR(5%) = ", VaR_10d)
 println("10-Dias ETL(5%) = ", ETL_10d)